In [ ]:
"""Configure the Pachner stress CLI run."""

import json
import os
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display as ipython_display

DIMENSION: int = 3
VERTEX_COUNT: int = 5
MOVE_ATTEMPTS: int = 2
VALIDATE_EVERY: int = 1

RUN_COUNT: int = 1
VERTEX_COUNT_STEP: int = 0
MOVE_ATTEMPTS_STEP: int = 0
SEED_STEP: int = 1

KEY_REFRESH_EVERY: int = 1
RETRY_ATTEMPTS: int = 4
BASE_SEED: int = 7
TIMEOUT_SECONDS: int = 120
FIGURE_DPI: int = 120
RUN_FRESH: bool = True
FORCE_REBUILD_BINARY: bool = False
QUIET_CLI: bool = True
OUTPUT_DIR: str = "target/notebooks/02_pachner_stress_cli"

# Pachner stress CLI telemetry

This notebook drives the opt-in `delaunay pachner-stress` binary, normalizes the CLI's CSV/JSON artifacts into Parquet tables with Polars, and plots from Polars `scan_parquet` queries. The first cell is the scalar run configuration; use `RUN_COUNT` with scalar step values for a small sweep.

The library API is not imported here; the binary is the execution boundary.

In [ ]:
"""Build validated run configs from the first-cell scalar variables."""

type StressDimension = Literal["3d", "4d"]


@dataclass(frozen=True, slots=True)
class StressRunConfig:
    """Configuration for one CLI-backed Pachner stress run."""

    run_id: int
    dimension: StressDimension
    vertices: int
    attempts: int
    validate_every: int
    key_refresh_every: int
    retry_attempts: int
    seed: int
    timeout_seconds: int
    output_root: Path
    quiet: bool

    @property
    def case(self) -> str:
        """Return a deterministic artifact label for this run."""
        return f"{self.run_id:02d}_{self.dimension}_{self.vertices}v_{self.attempts}moves_validate{self.validate_every}_seed{self.seed}"

    @property
    def run_dir(self) -> Path:
        """Return the artifact directory for this run."""
        return self.output_root / "runs" / self.case

    @property
    def progress_csv(self) -> Path:
        """Return the CLI progress CSV path for this run."""
        return self.run_dir / "progress.csv"

    @property
    def summary_json(self) -> Path:
        """Return the CLI summary JSON path for this run."""
        return self.run_dir / "summary.json"

    @property
    def stdout_log(self) -> Path:
        """Return the captured stdout log path for this run."""
        return self.run_dir / "stdout.log"

    @property
    def stderr_log(self) -> Path:
        """Return the captured stderr log path for this run."""
        return self.run_dir / "stderr.log"

    @property
    def command_json(self) -> Path:
        """Return the captured command metadata path for this run."""
        return self.run_dir / "command.json"


def find_repo_root(start: Path) -> Path:
    """Find the repository root from a notebook launch directory."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").is_file() and (path / "Cargo.toml").is_file():
            return path
    message = "Run this notebook from inside the delaunay repository."
    raise RuntimeError(message)


def parse_dimension(value: object) -> StressDimension:
    """Parse the supported Pachner stress notebook dimensions."""
    if type(value) is not int:
        raise TypeError(f"DIMENSION must be an integer, got {value!r}")
    if value == 3:
        return "3d"
    if value == 4:
        return "4d"
    message = f"DIMENSION must be 3 or 4, got {value}"
    raise ValueError(message)


def parse_positive_int(name: str, value: object) -> int:
    """Parse a positive integer run parameter."""
    if type(value) is not int:
        raise TypeError(f"{name} must be an integer, got {value!r}")
    if value <= 0:
        message = f"{name} must be positive, got {value}"
        raise ValueError(message)
    return value


def parse_non_negative_int(name: str, value: object) -> int:
    """Parse a non-negative integer run parameter."""
    if type(value) is not int:
        raise TypeError(f"{name} must be an integer, got {value!r}")
    if value < 0:
        message = f"{name} must be non-negative, got {value}"
        raise ValueError(message)
    return value


def parse_bool(name: str, value: object) -> bool:
    """Parse a boolean run parameter without truthiness coercion."""
    if type(value) is not bool:
        raise TypeError(f"{name} must be a boolean, got {value!r}")
    return value


def parse_output_root(value: object) -> Path:
    """Parse the notebook artifact root from a scalar path string."""
    if not isinstance(value, str):
        raise TypeError(f"OUTPUT_DIR must be a string, got {value!r}")
    if not value.strip():
        message = "OUTPUT_DIR must not be empty"
        raise ValueError(message)
    path = Path(value).expanduser()
    return path if path.is_absolute() else ROOT / path


def minimum_vertices(dimension: StressDimension) -> int:
    """Return the CLI minimum initial vertex count for one dimension."""
    return 4 if dimension == "3d" else 5


def build_stress_run_config(run_index: int, dimension: StressDimension, output_root: Path) -> StressRunConfig:
    """Build one validated CLI run config from scalar first-cell controls."""
    offset = run_index - 1
    vertices = parse_positive_int("VERTEX_COUNT + run offset", VERTEX_COUNT_BASE + offset * VERTEX_COUNT_STEP_PARSED)
    attempts = parse_positive_int("MOVE_ATTEMPTS + run offset", MOVE_ATTEMPTS_BASE + offset * MOVE_ATTEMPTS_STEP_PARSED)
    validate_every = VALIDATE_EVERY_PARSED
    minimum = minimum_vertices(dimension)
    if vertices < minimum:
        message = f"{dimension} needs at least {minimum} vertices, got {vertices}"
        raise ValueError(message)
    return StressRunConfig(
        run_id=run_index,
        dimension=dimension,
        vertices=vertices,
        attempts=attempts,
        validate_every=validate_every,
        key_refresh_every=KEY_REFRESH_EVERY_PARSED,
        retry_attempts=RETRY_ATTEMPTS_PARSED,
        seed=BASE_SEED_PARSED + offset * SEED_STEP_PARSED,
        timeout_seconds=TIMEOUT_SECONDS_PARSED,
        output_root=output_root,
        quiet=QUIET_CLI_PARSED,
    )


ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_ROOT = parse_output_root(OUTPUT_DIR)
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for directory in (OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DIMENSION_LABEL = parse_dimension(DIMENSION)
RUNS = parse_positive_int("RUN_COUNT", RUN_COUNT)
VERTEX_COUNT_BASE = parse_positive_int("VERTEX_COUNT", VERTEX_COUNT)
MOVE_ATTEMPTS_BASE = parse_positive_int("MOVE_ATTEMPTS", MOVE_ATTEMPTS)
VALIDATE_EVERY_PARSED = parse_positive_int("VALIDATE_EVERY", VALIDATE_EVERY)
VERTEX_COUNT_STEP_PARSED = parse_non_negative_int("VERTEX_COUNT_STEP", VERTEX_COUNT_STEP)
MOVE_ATTEMPTS_STEP_PARSED = parse_non_negative_int("MOVE_ATTEMPTS_STEP", MOVE_ATTEMPTS_STEP)
SEED_STEP_PARSED = parse_non_negative_int("SEED_STEP", SEED_STEP)
KEY_REFRESH_EVERY_PARSED = parse_positive_int("KEY_REFRESH_EVERY", KEY_REFRESH_EVERY)
RETRY_ATTEMPTS_PARSED = parse_positive_int("RETRY_ATTEMPTS", RETRY_ATTEMPTS)
BASE_SEED_PARSED = parse_non_negative_int("BASE_SEED", BASE_SEED)
TIMEOUT_SECONDS_PARSED = parse_positive_int("TIMEOUT_SECONDS", TIMEOUT_SECONDS)
FIGURE_DPI_PARSED = parse_positive_int("FIGURE_DPI", FIGURE_DPI)
RUN_FRESH_PARSED = parse_bool("RUN_FRESH", RUN_FRESH)
FORCE_REBUILD_BINARY_PARSED = parse_bool("FORCE_REBUILD_BINARY", FORCE_REBUILD_BINARY)
QUIET_CLI_PARSED = parse_bool("QUIET_CLI", QUIET_CLI)

CONFIGS = [build_stress_run_config(run_index, DIMENSION_LABEL, OUTPUT_ROOT) for run_index in range(1, RUNS + 1)]

plt.rcParams.update({"figure.dpi": FIGURE_DPI_PARSED})
print(f"Configured {len(CONFIGS)} Pachner stress CLI run(s).")
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
"""Resolve the binary and execute the configured CLI runs."""


def run_command(command: list[str], *, timeout_seconds: int) -> subprocess.CompletedProcess[str]:
    """Run one command from the repository root and capture its output."""
    return subprocess.run(  # noqa: S603 - argv list with shell=False and controlled cwd.
        command,
        cwd=ROOT,
        env=os.environ.copy(),
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
        check=False,
    )


def resolve_delaunay_binary() -> Path:
    """Return the CLI binary path, building it when no override is supplied."""
    override = os.environ.get("DELAUNAY_BINARY")
    if override is not None:
        if not override.strip():
            message = "DELAUNAY_BINARY must not be empty"
            raise ValueError(message)
        binary = Path(override).expanduser()
        binary = binary if binary.is_absolute() else ROOT / binary
        if not binary.is_file():
            message = f"DELAUNAY_BINARY does not exist: {binary}"
            raise FileNotFoundError(message)
        return binary

    binary = ROOT / "target" / "perf" / "delaunay"
    if FORCE_REBUILD_BINARY_PARSED or not binary.is_file():
        cargo = shutil.which("cargo")
        if cargo is None:
            message = "cargo executable was not found on PATH"
            raise RuntimeError(message)
        build = run_command(
            [cargo, "build", "--profile", "perf", "--features", "cli", "--bin", "delaunay"],
            timeout_seconds=TIMEOUT_SECONDS_PARSED,
        )
        if build.returncode != 0:
            message = "failed to build the delaunay CLI"
            details = f"{message}\nstdout:\n{build.stdout}\nstderr:\n{build.stderr}"
            raise RuntimeError(details)
    return binary


def pachner_command(binary: Path, config: StressRunConfig) -> list[str]:
    """Build the `delaunay pachner-stress` command for one config."""
    command = [
        str(binary),
        "pachner-stress",
        "--dimension",
        config.dimension,
        "--vertices",
        str(config.vertices),
        "--attempts",
        str(config.attempts),
        "--validate-every",
        str(config.validate_every),
        "--key-refresh-every",
        str(config.key_refresh_every),
        "--retry-attempts",
        str(config.retry_attempts),
        "--seed",
        str(config.seed),
        "--progress-csv",
        str(config.progress_csv),
        "--summary-json",
        str(config.summary_json),
    ]
    if config.quiet:
        command.append("--quiet")
    return command


def write_command_metadata(config: StressRunConfig, command: list[str], returncode: int) -> None:
    """Persist the command line used for reproducibility."""
    payload = {
        "case": config.case,
        "command": command,
        "returncode": returncode,
        "cwd": str(ROOT),
    }
    config.command_json.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")


def run_pachner_stress(binary: Path, config: StressRunConfig) -> None:
    """Run one CLI case and assert that expected artifacts were produced."""
    config.run_dir.mkdir(parents=True, exist_ok=True)
    command = pachner_command(binary, config)
    result = run_command(command, timeout_seconds=config.timeout_seconds)
    config.stdout_log.write_text(result.stdout, encoding="utf-8")
    config.stderr_log.write_text(result.stderr, encoding="utf-8")
    write_command_metadata(config, command, result.returncode)
    if result.returncode != 0:
        message = f"Pachner stress CLI failed for {config.case} with exit code {result.returncode}. See {config.stdout_log} and {config.stderr_log}."
        raise RuntimeError(message)
    for artifact in (config.progress_csv, config.summary_json):
        if not artifact.is_file():
            message = f"CLI did not write expected artifact: {artifact}"
            raise FileNotFoundError(message)


if RUN_FRESH_PARSED:
    DELAUNAY_BINARY = resolve_delaunay_binary()
    print(f"Using delaunay binary: {DELAUNAY_BINARY}")
    for run_config in CONFIGS:
        run_pachner_stress(DELAUNAY_BINARY, run_config)
        print(f"Wrote {run_config.run_dir}")
else:
    print("RUN_FRESH is false; loading existing artifacts for the configured cases.")

In [ ]:
"""Normalize CLI CSV/JSON artifacts into Polars-backed Parquet tables."""

SUMMARY_PARQUET = TABLE_DIR / "summary.parquet"
PROGRESS_PARQUET = TABLE_DIR / "progress.parquet"
PROGRESS_SCHEMA = {
    "dimension": pl.Int64,
    "label": pl.String,
    "sequence": pl.Int64,
    "step": pl.Int64,
    "attempts": pl.Int64,
    "accepted": pl.Int64,
    "rejected": pl.Int64,
    "candidate_misses": pl.Int64,
    "proposal_rejections": pl.Int64,
    "validations": pl.Int64,
    "validation_nanos": pl.Int64,
    "acceptance_rate": pl.Float64,
    "vertices": pl.Int64,
    "simplices": pl.Int64,
    "rss_kib": pl.Int64,
}


NON_NEGATIVE_PROGRESS_COLUMNS = [
    "sequence",
    "step",
    "attempts",
    "accepted",
    "rejected",
    "candidate_misses",
    "proposal_rejections",
    "validations",
    "validation_nanos",
    "vertices",
    "simplices",
    "rss_kib",
]


def reject_json_constant(value: str) -> object:
    """Reject non-standard JSON numeric constants before artifact parsing."""
    raise ValueError(f"JSON artifact contains non-finite value {value}")


def dimension_value(dimension: StressDimension) -> int:
    """Return the numeric CLI dimension for a parsed dimension label."""
    return 3 if dimension == "3d" else 4


def require_object(value: Any, context: str) -> dict[str, Any]:
    """Return a JSON object from a CLI artifact."""
    if not isinstance(value, dict):
        raise TypeError(f"{context} must be a JSON object")
    return value


def require_str(value: Any, context: str) -> str:
    """Return a JSON string from a CLI artifact."""
    if not isinstance(value, str):
        raise TypeError(f"{context} must be a string")
    return value


def require_non_negative_int(value: Any, context: str) -> int:
    """Return a non-negative JSON integer from a CLI artifact."""
    if type(value) is not int:
        raise TypeError(f"{context} must be an integer")
    if value < 0:
        raise ValueError(f"{context} must be non-negative, got {value}")
    return value


def require_positive_int(value: Any, context: str) -> int:
    """Return a positive JSON integer from a CLI artifact."""
    parsed = require_non_negative_int(value, context)
    if parsed == 0:
        raise ValueError(f"{context} must be positive")
    return parsed


def summary_record(config: StressRunConfig) -> dict[str, int | float | str]:
    """Flatten one summary JSON artifact into a validated table row."""
    data = require_object(
        json.loads(config.summary_json.read_text(encoding="utf-8"), parse_constant=reject_json_constant),
        str(config.summary_json),
    )
    source = require_object(data.get("source"), "summary.source")
    report = require_object(data.get("report"), "summary.report")
    attempts = require_positive_int(report.get("attempts"), "summary.report.attempts")
    elapsed_nanos = require_non_negative_int(report.get("elapsed_nanos"), "summary.report.elapsed_nanos")
    validation_nanos = require_non_negative_int(report.get("validation_nanos"), "summary.report.validation_nanos")
    accepted = require_non_negative_int(report.get("accepted"), "summary.report.accepted")
    rejected = require_non_negative_int(report.get("rejected"), "summary.report.rejected")
    if accepted + rejected != attempts:
        raise ValueError(f"summary.report accepted + rejected must equal attempts for {config.case}")
    summary_dimension = require_positive_int(data.get("dimension"), "summary.dimension")
    summary_label = require_str(data.get("label"), "summary.label")
    configured_vertices = require_positive_int(data.get("configured_vertices"), "summary.configured_vertices")
    configured_attempts = require_positive_int(data.get("attempts"), "summary.attempts")
    if summary_dimension != dimension_value(config.dimension):
        raise ValueError(f"summary.dimension must match {config.dimension} for {config.case}")
    if summary_label != config.dimension:
        raise ValueError(f"summary.label must be {config.dimension!r} for {config.case}")
    if configured_vertices != config.vertices:
        raise ValueError(f"summary.configured_vertices must match config vertices for {config.case}")
    if configured_attempts != config.attempts:
        raise ValueError(f"summary.attempts must match config attempts for {config.case}")
    return {
        "run_id": config.run_id,
        "case": config.case,
        "dimension": summary_dimension,
        "label": summary_label,
        "configured_vertices": configured_vertices,
        "attempts": configured_attempts,
        "validate_every": require_positive_int(data.get("validate_every"), "summary.validate_every"),
        "key_refresh_every": require_positive_int(data.get("key_refresh_every"), "summary.key_refresh_every"),
        "retry_attempts": require_positive_int(data.get("retry_attempts"), "summary.retry_attempts"),
        "min_vertex_count": require_positive_int(data.get("min_vertex_count"), "summary.min_vertex_count"),
        "max_vertex_count": require_positive_int(data.get("max_vertex_count"), "summary.max_vertex_count"),
        "seed": require_non_negative_int(data.get("seed"), "summary.seed"),
        "source_vertices": require_positive_int(source.get("vertices"), "summary.source.vertices"),
        "source_simplices": require_non_negative_int(source.get("simplices"), "summary.source.simplices"),
        "accepted": accepted,
        "rejected": rejected,
        "candidate_misses": require_non_negative_int(report.get("candidate_misses"), "summary.report.candidate_misses"),
        "proposal_rejections": require_non_negative_int(report.get("proposal_rejections"), "summary.report.proposal_rejections"),
        "validations": require_non_negative_int(report.get("validations"), "summary.report.validations"),
        "validation_seconds": validation_nanos / 1_000_000_000,
        "elapsed_seconds": elapsed_nanos / 1_000_000_000,
        "attempts_per_second": require_non_negative_int(report.get("attempts_per_second"), "summary.report.attempts_per_second"),
        "acceptance_fraction": accepted / attempts,
        "final_vertices": require_positive_int(report.get("final_vertices"), "summary.report.final_vertices"),
        "final_simplices": require_non_negative_int(report.get("final_simplices"), "summary.report.final_simplices"),
        "start_rss_kib": require_non_negative_int(report.get("start_rss_kib"), "summary.report.start_rss_kib"),
        "max_rss_kib": require_non_negative_int(report.get("max_rss_kib"), "summary.report.max_rss_kib"),
        "final_rss_kib": require_non_negative_int(report.get("final_rss_kib"), "summary.report.final_rss_kib"),
    }


def progress_frame(config: StressRunConfig) -> pl.DataFrame:
    """Read and validate one CLI progress CSV artifact with Polars."""
    frame = pl.read_csv(config.progress_csv, schema_overrides=PROGRESS_SCHEMA)
    missing_columns = [column for column in PROGRESS_SCHEMA if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{config.progress_csv} is missing columns: {missing_columns}")
    null_counts = frame.select(pl.all().null_count()).row(0)
    null_columns = [column for column, count in zip(frame.columns, null_counts, strict=True) if count]
    if null_columns:
        raise ValueError(f"{config.progress_csv} contains nulls in columns: {null_columns}")
    if frame.is_empty():
        return frame.with_columns(
            pl.lit(config.run_id).alias("run_id"),
            pl.lit(config.case).alias("case"),
        )
    negative_columns = [column for column in NON_NEGATIVE_PROGRESS_COLUMNS if frame.select((pl.col(column) < 0).any()).item()]
    if negative_columns:
        raise ValueError(f"{config.progress_csv} contains negative values in columns: {negative_columns}")
    invalid_dimension = frame.select((pl.col("dimension") != dimension_value(config.dimension)).any()).item()
    if invalid_dimension:
        raise ValueError(f"{config.progress_csv} contains rows for the wrong dimension")
    invalid_label = frame.select((pl.col("label") != config.dimension).any()).item()
    if invalid_label:
        raise ValueError(f"{config.progress_csv} contains rows for the wrong dimension label")
    invalid_attempts = frame.select((pl.col("attempts") != config.attempts).any()).item()
    if invalid_attempts:
        raise ValueError(f"{config.progress_csv} contains rows for the wrong attempt count")
    invalid_progress_counts = frame.select(
        (
            (pl.col("step") > pl.col("attempts"))
            | ((pl.col("accepted") + pl.col("rejected")) > pl.col("step"))
            | (pl.col("accepted") > pl.col("attempts"))
            | (pl.col("rejected") > pl.col("attempts"))
        ).any()
    ).item()
    if invalid_progress_counts:
        raise ValueError(f"{config.progress_csv} has progress counters outside configured attempts")
    invalid_acceptance_rate = frame.select(
        (~pl.col("acceptance_rate").is_finite() | (pl.col("acceptance_rate") < 0.0) | (pl.col("acceptance_rate") > 1.0)).any()
    ).item()
    if invalid_acceptance_rate:
        raise ValueError(f"{config.progress_csv} contains acceptance_rate outside [0, 1]")
    return frame.with_columns(
        pl.lit(config.run_id).alias("run_id"),
        pl.lit(config.case).alias("case"),
    )


def write_current_parquet_scan(frame: pl.DataFrame, path: Path) -> pl.LazyFrame:
    """Write the current table and return a scan, removing stale empty outputs."""
    if frame.is_empty():
        if path.exists():
            if not path.is_file():
                raise IsADirectoryError(f"expected Parquet file path, got directory: {path}")
            path.unlink()
        return pl.LazyFrame()
    frame.write_parquet(path)
    return pl.scan_parquet(path)


available_configs = [config for config in CONFIGS if config.summary_json.is_file() and config.progress_csv.is_file()]
missing_configs = [config.case for config in CONFIGS if config not in available_configs]
if missing_configs:
    print(f"Missing artifacts for {len(missing_configs)} configured case(s): {missing_configs}")

summary_rows = [summary_record(config) for config in available_configs]
summary_artifact_df = pl.DataFrame(summary_rows) if summary_rows else pl.DataFrame()
progress_frames = [progress_frame(config) for config in available_configs]
progress_artifact_df = pl.concat(progress_frames, how="diagonal") if progress_frames else pl.DataFrame()

summary_lf = write_current_parquet_scan(summary_artifact_df, SUMMARY_PARQUET)
progress_lf = write_current_parquet_scan(progress_artifact_df, PROGRESS_PARQUET)
summary_df = summary_lf.collect()
progress_df = progress_lf.collect()

print(f"Summary rows: {summary_df.height}; progress rows: {progress_df.height}")
print(f"Polars Parquet scans: {SUMMARY_PARQUET}, {PROGRESS_PARQUET}")

In [ ]:
"""Show compact source and final metric tables from Polars Parquet scans."""

if summary_df.is_empty():
    print("No Pachner stress artifacts are available for the configured cases.")
else:
    summary_columns = [
        "case",
        "dimension",
        "configured_vertices",
        "attempts",
        "accepted",
        "rejected",
        "acceptance_fraction",
        "attempts_per_second",
        "final_vertices",
        "final_simplices",
        "max_rss_kib",
    ]
    ipython_display(summary_lf.sort("run_id").select(summary_columns).collect())

if progress_df.is_empty():
    print("No progress rows are available for the configured cases.")
else:
    progress_columns = [
        "case",
        "step",
        "accepted",
        "rejected",
        "acceptance_rate",
        "vertices",
        "simplices",
        "rss_kib",
    ]
    ipython_display(progress_lf.select(progress_columns).sort(["case", "step"]).tail(12).collect())

In [ ]:
"""Plot progress traces for accepted moves, topology size, and memory."""


def save_figure(name: str) -> None:
    """Save the current Matplotlib figure into the notebook artifact directory."""
    path = FIGURE_DIR / name
    plt.savefig(path, bbox_inches="tight")
    print(f"Wrote {path}")


if progress_df.is_empty():
    print("No progress telemetry to plot yet.")
else:
    progress = progress_lf.sort(["case", "step"]).collect()
    fig, axes = plt.subplots(3, 1, figsize=(9.5, 8.5), sharex=True, layout="constrained")
    for case in progress["case"].unique().sort().to_list():
        trace = progress.filter(pl.col("case") == case)
        steps = trace["step"].to_numpy()
        axes[0].plot(steps, trace["accepted"].to_numpy(), label=f"accepted {case}")
        axes[0].plot(steps, trace["rejected"].to_numpy(), linestyle="--", label=f"rejected {case}")
        axes[1].plot(steps, trace["vertices"].to_numpy(), label=f"vertices {case}")
        axes[1].plot(steps, trace["simplices"].to_numpy(), linestyle="--", label=f"simplices {case}")
        axes[2].plot(steps, trace["rss_kib"].to_numpy(), label=f"RSS {case}")

    axes[0].set_ylabel("moves")
    axes[1].set_ylabel("count")
    axes[2].set_ylabel("RSS KiB")
    axes[2].set_xlabel("attempted moves")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(loc="best", fontsize="x-small")
    fig.suptitle("Pachner stress progress by CLI case")
    save_figure("progress_traces.png")

In [ ]:
"""Plot final throughput and acceptance diagnostics."""

if summary_df.is_empty():
    print("No final metric telemetry to plot yet.")
else:
    summary = summary_lf.sort("run_id").collect()
    labels = summary["case"].to_list()
    positions = range(len(labels))

    fig, axes = plt.subplots(3, 1, figsize=(9.5, 8.5), layout="constrained")
    axes[0].bar(positions, summary["attempts_per_second"].to_list())
    axes[0].set_ylabel("attempts/s")
    axes[0].set_title("throughput")

    axes[1].bar(positions, summary["acceptance_fraction"].to_list())
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_ylabel("accepted / attempts")
    axes[1].set_title("acceptance")

    axes[2].bar(positions, summary["final_vertices"].to_list(), label="vertices")
    axes[2].bar(positions, summary["final_simplices"].to_list(), alpha=0.65, label="simplices")
    axes[2].set_ylabel("final count")
    axes[2].set_title("final topology size")
    axes[2].legend(loc="best", fontsize="small")

    for axis in axes:
        axis.set_xticks(list(positions))
        axis.set_xticklabels(labels, rotation=20, ha="right")
        axis.grid(axis="y", alpha=0.25)
    fig.suptitle("Pachner stress final metrics")
    save_figure("final_metrics.png")